# ComplaintLens: Complaint Risk Analysis

This notebook is the reviewable analytical companion to the production pipeline. It uses the versioned aggregate marts, validates the headline KPI, and recreates the decision views without loading the 5.4 GB raw CSV.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed'
monthly = pd.read_csv(DATA / 'fact_monthly.csv')
products = pd.read_csv(DATA / 'fact_product_monthly.csv')
companies = pd.read_csv(DATA / 'fact_company_product.csv')
issues = pd.read_csv(DATA / 'fact_issue_product.csv')

## 1. KPI reconciliation

The aggregate marts should reconcile to the executive KPI contract before analysis begins.

In [ ]:
kpis = pd.read_csv(DATA / 'executive_kpis.csv').iloc[0]
assert monthly['complaint_count'].sum() == kpis['complaint_count']
assert monthly['untimely_count'].sum() == kpis['untimely_count']
kpis.to_frame('value')

## 2. Demand and service trend

Volume more than doubled from January to December while the untimely-response rate declined. This suggests that portfolio capacity held, but it does not rule out segment-level exceptions.

In [ ]:
monthly['untimely_rate'] = 1 - monthly['timely_rate']
ax = monthly.plot(x='month', y='complaint_count', marker='o', figsize=(11, 5), color='#0077B6')
ax.set(title='2024 complaint workload', xlabel='', ylabel='Complaints')
plt.xticks(rotation=45)
plt.tight_layout()

## 3. Product risk

A useful triage view keeps workload and exception rate separate. Credit reporting dominates volume, while smaller products such as student loans and prepaid cards have higher untimely rates.

In [ ]:
product_summary = products.groupby('product', as_index=False)[['complaint_count','timely_count','untimely_count']].sum()
product_summary['untimely_rate'] = product_summary['untimely_count'] / (product_summary['timely_count'] + product_summary['untimely_count'])
product_summary.sort_values('untimely_count', ascending=False).head(10)

## 4. Company-product triage

Only queues with at least 100 complaints are compared. Ranking uses untimely count rather than an opaque composite score.

In [ ]:
columns = ['company','product','complaint_count','untimely_count','timely_rate','additional_timely_for_99pct']
companies.loc[companies['company_comparison_eligible'], columns].sort_values('untimely_count', ascending=False).head(15)

## 5. Interpretation guardrail

Complaint volume is an operational workload signal, not a market-adjusted company-quality score. The dataset is not a statistical sample and does not include customer counts or market share. These results identify where to investigate; they do not prove causality or wrongdoing.